# Практика 4. Метрики и валидация

**Курс «Машинное обучение» · Часть 1, занятие 4**

Самое важное занятие первой части курса. Разбираемся, как честно измерять качество.

План:

1. Почему accuracy обманывает — на данных с дисбалансом
2. ROC и PR-кривые: в чём разница и когда какая
3. Порог решения как бизнес-параметр
4. Схемы валидации и две ловушки: группы и время
5. Подбор гиперпараметров случайным поиском
6. Если классов больше двух

Всё занятие работаем с логистической регрессией — модель знакомая, а разговор сегодня не про неё, а про то, как её оценивать.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import loguniform
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_val_predict, RandomizedSearchCV,
    StratifiedKFold, KFold, GroupKFold, TimeSeriesSplit,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("готово")

---
## 1. Данные с дисбалансом

Сделаем задачу, похожую на реальную: положительный класс встречается в 3% случаев. Так выглядят мошенничество, отказы оборудования, редкие болезни.

In [ ]:
X, y = make_classification(
    n_samples=10000, n_features=20, n_informative=6, n_redundant=4,
    weights=[0.97, 0.03],      # 3% положительного класса
    flip_y=0.02, class_sep=0.8,
    random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"обучающая: {X_train.shape[0]}, тестовая: {X_test.shape[0]}")
print(f"доля положительного класса: {y.mean():.3f}")
print(f"положительных в тесте: {y_test.sum()} из {len(y_test)}")

In [ ]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print("Выглядит прекрасно. Но посмотрим на глупую модель, которая")
print("вообще ничего не предсказывает:")
print(f"accuracy «всё отрицательное»: {(y_test == 0).mean():.4f}")

**Вот оно.** Модель, которая всегда отвечает «нет», даёт почти такую же accuracy. Метрика не различает работающую модель и бесполезную.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["предск. 0", "предск. 1"],
            yticklabels=["истина 0", "истина 1"], ax=ax)
ax.set_title("Матрица ошибок")
plt.tight_layout()
plt.show()

print(f"верно найдено положительных (TP): {tp}")
print(f"пропущено положительных (FN):    {fn}")
print(f"ложных тревог (FP):              {fp}")
print()
print(f"precision: {precision_score(y_test, y_pred):.3f}  "
      f"— какая доля тревог настоящая")
print(f"recall:    {recall_score(y_test, y_pred):.3f}  "
      f"— какую долю положительных поймали")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print()
print(f"Модель нашла только {tp} из {tp + fn} положительных объектов.")
print("Accuracy этого не показала.")

---
## 2. ROC и PR-кривые

Обе показывают модель целиком, при всех возможных порогах. Но ведут себя по-разному.

In [ ]:
fpr, tpr, roc_thr = roc_curve(y_test, y_proba)
precision, recall, pr_thr = precision_recall_curve(y_test, y_proba)

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
baseline_pr = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color="#2B5CE6", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="#A8B0BC", label="случайная модель")
axes[0].set_xlabel("доля ложных тревог (FPR)")
axes[0].set_ylabel("доля пойманных (TPR)")
axes[0].set_title("ROC-кривая")
axes[0].legend()

axes[1].plot(recall, precision, color="#1B8A5A", lw=2, label=f"AP = {pr_auc:.3f}")
axes[1].axhline(baseline_pr, ls="--", color="#A8B0BC",
                label=f"случайная модель = {baseline_pr:.3f}")
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("PR-кривая")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc_auc:.3f}  — выглядит отлично")
print(f"PR-AUC:  {pr_auc:.3f}  — куда скромнее")
print()
print("Причина: ROC-кривая использует долю ложных тревог, а знаменатель")
print("там — все отрицательные объекты, которых у нас 97%. Сотня ложных")
print("тревог на фоне 2900 отрицательных почти не двигает FPR.")
print("Но на фоне 90 положительных эта сотня рушит precision.")

### Задание 1

Убедитесь, что разрыв между метриками действительно связан с дисбалансом: посчитайте обе метрики на исходной задаче и на сбалансированной.

Сохраните: `roc_imbalanced` и `pr_imbalanced` — метрики на несбалансированной задаче, `roc_balanced` и `pr_balanced` — на сбалансированной.

> **Что посмотреть:** `roc_auc_score(y, proba)` и `average_precision_score(y, proba)` — обе принимают **вероятности**, а не метки.

In [ ]:
def evaluate_balance(weights, label):
    """Генерирует задачу с заданным балансом и считает обе метрики."""
    Xb, yb = make_classification(
        n_samples=10000, n_features=20, n_informative=6, n_redundant=4,
        weights=weights, flip_y=0.02, class_sep=0.8,
        random_state=RANDOM_STATE,
    )
    Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
        Xb, yb, test_size=0.3, random_state=RANDOM_STATE, stratify=yb
    )
    m = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    proba = m.fit(Xb_tr, yb_tr).predict_proba(Xb_te)[:, 1]

    # ── ВАШ КОД (две строки) ──────────────────────────────────────
    # Посчитайте обе метрики. Обе принимают ВЕРОЯТНОСТИ, а не метки
    roc = ...
    ap = ...
    # ──────────────────────────────────────────────────────────────

    print(f"{label:22} доля класса 1 = {yb_te.mean():.3f}   "
          f"ROC-AUC = {roc:.3f}   PR-AUC = {ap:.3f}   разрыв = {roc - ap:+.3f}")
    return roc, ap


roc_imbalanced, pr_imbalanced = evaluate_balance([0.97, 0.03], "дисбаланс 3%")
roc_balanced, pr_balanced = evaluate_balance([0.5, 0.5], "сбалансированная")


In [ ]:
# Проверка задания 1
assert round(float(roc_imbalanced), 2) == 0.73, f"roc_imbalanced = {roc_imbalanced}"
assert round(float(pr_imbalanced), 2) == 0.29, f"pr_imbalanced = {pr_imbalanced}"
assert round(float(roc_balanced), 2) == 0.83, f"roc_balanced = {roc_balanced}"
assert round(float(pr_balanced), 2) == 0.84, f"pr_balanced = {pr_balanced}"
assert roc_imbalanced - pr_imbalanced > roc_balanced - pr_balanced,     "расхождение метрик при дисбалансе должно быть больше"
print("Задание 1 — верно ✓")

---
## 3. Порог решения

`predict()` использует порог 0.5. Это произвольное число. Подберём его двумя способами — по метрике и по деньгам — и сравним.

In [ ]:
COST_FN, COST_FP = 500, 20   # пропуск стоит в 25 раз дороже ложной тревоги

thresholds = np.linspace(0.01, 0.9, 90)
precisions, recalls, f1s, costs = [], [], [], []
for t in thresholds:
    tn, fp, fn, tp = confusion_matrix(y_test, (y_proba >= t).astype(int)).ravel()
    precisions.append(tp / max(tp + fp, 1))
    recalls.append(tp / (tp + fn))
    f1s.append(2 * tp / max(2 * tp + fp + fn, 1))
    costs.append(fn * COST_FN + fp * COST_FP)

f1s, costs = np.array(f1s), np.array(costs)
best_f1_t = thresholds[f1s.argmax()]
best_cost_t = thresholds[costs.argmin()]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for values, color, label in [(precisions, "#2B5CE6", "precision"),
                             (recalls, "#C0393B", "recall"),
                             (f1s, "#1B8A5A", "F1")]:
    axes[0].plot(thresholds, values, color=color, label=label)
axes[0].axvline(best_f1_t, ls="--", color="#6B7484", label=f"лучший F1: {best_f1_t:.2f}")
axes[0].set_title("Выбор порога по метрике")

axes[1].plot(thresholds, costs, color="#2B5CE6")
axes[1].axvline(best_cost_t, ls="--", color="#C0393B",
                label=f"минимум потерь: {best_cost_t:.2f}")
axes[1].set_ylabel("суммарные потери")
axes[1].set_title("Выбор порога по цене ошибок")

for ax in axes:
    ax.axvline(0.5, ls=":", color="#A8B0BC", label="порог по умолчанию")
    ax.set_xlabel("порог")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

cost_default = costs[np.argmin(np.abs(thresholds - 0.5))]
print(f"F1:     порог 0.5 → {f1s[np.argmin(np.abs(thresholds - 0.5))]:.3f}, "
      f"порог {best_f1_t:.2f} → {f1s.max():.3f}")
print(f"потери: порог 0.5 → {cost_default:,.0f}, "
      f"порог {best_cost_t:.2f} → {costs.min():,.0f}  "
      f"(экономия {cost_default - costs.min():,.0f})")
print()
print("Два порога не совпали: по деньгам выгодно срабатывать намного чаще,")
print("чем советует F1. Метрика про баланс, а бизнес про цену ошибки.")

**Ключевая мысль:** порог выбирается не по метрике, а по цене ошибок. F1 неявно предполагает, что precision и recall одинаково важны — а это почти никогда не так.

---
## 4. Схемы валидации

Кросс-валидация честна ровно настолько, насколько её схема повторяет реальную работу модели. Две классические ловушки.

In [ ]:
# ЛОВУШКА 1: записи одного клиента почти совпадают.
# На каждого клиента приходится несколько похожих заявок
rng = np.random.default_rng(RANDOM_STATE)
n_clients, per_client, n_feat = 120, 5, 40

portrait = rng.normal(size=(n_clients, n_feat))        # «портрет» клиента
w = rng.normal(size=n_feat)
y_client = (portrait @ w + rng.normal(0, 1.0, n_clients) > 0).astype(int)

# Заявки клиента — его портрет с небольшим шумом
Xg = (np.repeat(portrait, per_client, axis=0)
      + rng.normal(0, 0.25, (n_clients * per_client, n_feat)))
yg = np.repeat(y_client, per_client)
clients = np.repeat(np.arange(n_clients), per_client)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
naive = cross_val_score(pipe, Xg, yg, scoring="roc_auc",
                        cv=StratifiedKFold(5, shuffle=True,
                                           random_state=RANDOM_STATE))
grouped = cross_val_score(pipe, Xg, yg, scoring="roc_auc",
                          cv=GroupKFold(5), groups=clients)

print(f"обычная кросс-валидация: {naive.mean():.3f}")
print(f"GroupKFold:              {grouped.mean():.3f}")
print()
print("Обычная схема разбрасывает заявки одного клиента между обучением")
print("и проверкой — почти такая же запись оказывается по обе стороны.")
print("GroupKFold держит каждого клиента целиком в одной части.")

In [ ]:
# ЛОВУШКА 2: объекты упорядочены во времени, а среда меняется
rng = np.random.default_rng(RANDOM_STATE)
n = 3000
drift = np.linspace(-2.5, 2.5, n)      # условия плавно меняются

Xt = rng.normal(size=(n, 5))
Xt[:, 0] += drift
yt = (Xt @ np.array([1.2, -0.8, 0.5, 0.0, 0.0])
      + drift + rng.normal(0, 1, n) > 0).astype(int)

shuffled = cross_val_score(pipe, Xt, yt, scoring="roc_auc",
                           cv=KFold(5, shuffle=True,
                                    random_state=RANDOM_STATE))
in_time = cross_val_score(pipe, Xt, yt, scoring="roc_auc",
                          cv=TimeSeriesSplit(5))

print(f"перемешанный KFold: {shuffled.mean():.3f}  ← учимся на будущем")
print(f"TimeSeriesSplit:    {in_time.mean():.3f}  ← учимся только на прошлом")
print(f"по срезам времени:  {np.round(in_time, 3)}")
print()
print("Перемешивание даёт модели заглянуть вперёд, и оценка выходит")
print("завышенной. В реальности будущего у неё не будет.")

### Задание 2

Третья ловушка проще и встречается чаще двух предыдущих — разбиение без стратификации. Сравните `KFold` и `StratifiedKFold` на нашей задаче с дисбалансом: посмотрите, как в каждой схеме распределены классы по фолдам.

Сохраните разброс долей положительного класса между фолдами в `share_std_kfold` и `share_std_stratified`.

> **Что посмотреть:** `cv_scheme.split(X_train, y_train)` выдаёт пары индексов `(обучающие, проверочные)`. Доля положительных в фолде — `y_train[test].mean()`, разброс — `np.std(список_долей)`.

In [ ]:
schemes = {
    "KFold": KFold(5, shuffle=True, random_state=RANDOM_STATE),
    "StratifiedKFold": StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
}

fold_shares = {}
for name, cv_scheme in schemes.items():

    # ── ВАШ КОД (одна строка) ─────────────────────────────────────
    # Соберите доли положительного класса по проверочным фолдам
    shares = ...
    # ──────────────────────────────────────────────────────────────

    scores = cross_val_score(model, X_train, y_train, cv=cv_scheme,
                             scoring="average_precision")
    fold_shares[name] = float(np.std(shares))
    print(f"{name}")
    print(f"   PR-AUC по фолдам:       {np.round(scores, 4)}")
    print(f"   доля класса 1 в фолдах: {np.round(shares, 4)}")
    print(f"   разброс долей:          {fold_shares[name]:.5f}")
    print()

share_std_kfold = fold_shares["KFold"]
share_std_stratified = fold_shares["StratifiedKFold"]


In [ ]:
# Проверка задания 2
assert round(float(share_std_kfold), 4) == 0.0025, f"share_std_kfold = {share_std_kfold}"
assert round(float(share_std_stratified), 4) == 0.0003,     f"share_std_stratified = {share_std_stratified}"
assert share_std_stratified < share_std_kfold,     "стратификация обязана выравнивать доли классов между фолдами"
print("Задание 2 — верно ✓")

---
## 5. Подбор гиперпараметров

У логистической регрессии настраивать особо нечего: сила регуляризации `C`, вес классов и степень полиномиальных признаков. Переберём их случайным поиском.

In [ ]:
search_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=1, include_bias=False)),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

# Для шагов пайплайна имя параметра пишется через двойное подчёркивание
space = {
    "poly__degree": [1, 2],
    "clf__C": loguniform(1e-3, 1e3),       # не список, а распределение
    "clf__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    search_pipe, space, n_iter=20, cv=cv, scoring="average_precision",
    random_state=RANDOM_STATE, n_jobs=-1,
)
search.fit(X_train, y_train)

print(f"PR-AUC модели по умолчанию: {pr_auc:.4f}")
print(f"PR-AUC лучшей на кросс-валидации: {search.best_score_:.4f}")
print("лучшие параметры:")
for k, v in search.best_params_.items():
    print(f"   {k}: {v}")

### Задание 3

Доведите работу до конца. Модель настроена, осталось выбрать порог — и сделать это, **не трогая тест**:

1. Получите out-of-fold предсказания на обучающей выборке
2. Подберите по ним порог, дающий лучший F1
3. Примените этот порог к тесту — один раз

Сохраните: `best_t` — подобранный порог, `f1_oof` — лучший F1 на out-of-fold предсказаниях, `f1_test` — F1 на тесте с этим порогом.

> **Что посмотреть:** `cross_val_predict(модель, X, y, cv=..., method="predict_proba")` даёт предсказания от моделей, которые этот объект не видели. Сетка порогов — `np.linspace(0.02, 0.8, 100)`, метки — `(proba >= t).astype(int)`, лучший порог — по `np.argmax` от списка F1.

In [ ]:
best_model = search.best_estimator_

# ── ВАШ КОД (две строки) ──────────────────────────────────────────
# 1. Получите ЧЕСТНЫЕ предсказания на обучающей выборке: каждый объект
#    предсказывается моделью, которая его не видела
oof_proba = ...
# 2. Посчитайте F1 для каждого порога из сетки ths
ths = np.linspace(0.02, 0.8, 100)
f1s_oof = ...
# ──────────────────────────────────────────────────────────────────

best_t = ths[int(np.argmax(f1s_oof))]
f1_oof = max(f1s_oof)

# Единственное обращение к тесту — уже с готовым порогом
test_proba = best_model.predict_proba(X_test)[:, 1]
f1_test = f1_score(y_test, (test_proba >= best_t).astype(int), zero_division=0)

print(f"порог, подобранный на кросс-валидации: {best_t:.3f}")
print(f"F1 на out-of-fold предсказаниях:       {f1_oof:.4f}")
print(f"F1 на тесте с этим порогом:            {f1_test:.4f}")
print()
print(f"{'модель':38} {'PR-AUC':>8} {'F1':>8}")
for name, proba, thr in [("логрег по умолчанию (порог 0.5)", y_proba, 0.5),
                         ("настроенный логрег (порог 0.5)", test_proba, 0.5),
                         ("настроенный + порог из CV", test_proba, best_t)]:
    pred = (proba >= thr).astype(int)
    print(f"{name:38} {average_precision_score(y_test, proba):8.4f} "
          f"{f1_score(y_test, pred, zero_division=0):8.4f}")


In [ ]:
# Проверка задания 3
assert round(float(best_t), 2) == 0.32, f"best_t = {best_t}"
assert round(float(f1_oof), 2) == 0.59, f"f1_oof = {f1_oof}"
assert round(float(f1_test), 2) == 0.61, f"f1_test = {f1_test}"
assert f1_test > 0.4, "настроенная модель с подобранным порогом должна побить F1 = 0.19"
print("Задание 3 — верно ✓")

> **Когда перебор становится дорогим.** Случайный поиск перебирает точки вслепую: прошлые результаты на следующую попытку не влияют. Если обучение занимает минуты, а параметров десяток, берут байесовскую оптимизацию — например, библиотеку **Optuna**: она выбирает следующую точку, глядя на уже испытанные. Для логистической регрессии на десяти тысячах объектов разницы не будет, а вот для градиентного бустинга в домашней работе — пригодится.

---
## 6. Если классов больше двух

Precision и recall считаются для каждого класса отдельно, а дальше их нужно собрать в одно число — и способ сборки меняет ответ.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report

digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    digits.data, digits.target, test_size=0.3,
    random_state=RANDOM_STATE, stratify=digits.target
)
# Специально портим баланс: оставляем мало примеров цифры 8
mask = ~((yd_tr == 8) & (np.arange(len(yd_tr)) % 5 != 0))
Xd_tr, yd_tr = Xd_tr[mask], yd_tr[mask]

clf = LogisticRegression(max_iter=5000).fit(Xd_tr, yd_tr)

print(f"примеров цифры 8 в обучении: {(yd_tr == 8).sum()}, "
      f"остальных цифр — около {(yd_tr == 0).sum()}")
print()
print(classification_report(yd_te, clf.predict(Xd_te), digits=3))
print("macro avg — среднее по классам, каждый класс весит одинаково")
print("weighted avg — среднее с весом по размеру класса")
print()
print("Если редкий класс важен, смотреть надо на macro: weighted его")
print("растворит в больших классах и покажет красивую картинку.")

---
## Итоги занятия

- **Accuracy при дисбалансе бесполезна** — сравнивайте с константной моделью
- ROC-AUC и PR-AUC отвечают на разные вопросы; при редком положительном классе смотрите PR-AUC
- Порог — параметр системы, и выбирать его нужно по цене ошибок, а не по умолчанию
- Схема валидации обязана повторять то, как модель будет работать в реальности: группы держим целиком, время не перемешиваем
- При нескольких классах способ усреднения меняет картину: macro и weighted отвечают на разные вопросы
- **Всё, что подбирается — гиперпараметры, порог, признаки — подбирается на обучающих данных**

## Домашнее задание 2

Выдаётся сегодня: линейные модели своей реализации плюс честная валидация.